# Clothes segmentation: the merged two-encoder model

Trains `dual_b3_convnext_tiny` (SegFormer-B3 encoder + ConvNeXt-Tiny encoder, fusion decoder) for 12 epochs and compares it with the single models. The rule is fixed in PLAN.md, section 32.3.

Use the **L4 GPU**. Run the cells from top to bottom; cell 3 asks for `clothseg_code.zip` once. If the session ends, run all cells again: the training continues from the last finished epoch saved on Drive.

## 1. Settings

In [3]:
REPO_URL = ""

DRIVE_ROOT = "/content/drive/MyDrive/clothseg"
CODE_DIR = "/content/clothes-segmentation"
CODE_VERSION = "14"
HEIGHT, WIDTH = 576, 384
SESSION_BUDGET_MIN = 600

## 2. Mount Drive and show the environment

In [4]:
import os, platform, shutil, time
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
SESSION_START = time.time()

def minutes_left():
    """Minutes of the session budget that remain; passed to every training command."""
    return max(1, SESSION_BUDGET_MIN - (time.time() - SESSION_START) / 60)

import torch, transformers
print("python      ", platform.python_version(), "| torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU         ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE: change the runtime type to a GPU")
if torch.cuda.is_available():
    print("GPU memory  ", f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print("CPU cores   ", os.cpu_count())
print("local disk  ", f"{shutil.disk_usage('/content').free / 1024**3:.0f} GB free")
print("Drive       ", f"{shutil.disk_usage(DRIVE_ROOT).free / 1024**3:.0f} GB free")
assert torch.cuda.is_available(), "No GPU. Choose Runtime > Change runtime type > L4 GPU, then run again."

Mounted at /content/drive
python       3.13.15 | torch 2.11.0+cu130 | transformers 5.18.0
GPU          NVIDIA L4
GPU memory   22.0 GB
CPU cores    12
local disk   193 GB free
Drive        184 GB free


## 3. Get the code

In [5]:
import subprocess, zipfile

def code_version(folder):
    path = os.path.join(folder, "VERSION")
    return open(path).read().strip() if os.path.exists(path) else "1"

os.chdir("/content")
if REPO_URL:
    if os.path.isdir(f"{CODE_DIR}/.git"):
        print(subprocess.run(["git", "-C", CODE_DIR, "pull"], capture_output=True, text=True).stdout)
    else:
        r = subprocess.run(["git", "clone", REPO_URL, CODE_DIR], capture_output=True, text=True)
        assert r.returncode == 0, f"git clone failed. Is the URL correct and the repository public?\n{r.stderr}"
elif os.path.isdir(f"{DRIVE_ROOT}/code") and code_version(f"{DRIVE_ROOT}/code") == CODE_VERSION:
    shutil.rmtree(CODE_DIR, ignore_errors=True)
    shutil.copytree(f"{DRIVE_ROOT}/code", CODE_DIR)
    print(f"code version {CODE_VERSION} copied from Drive")
else:
    from google.colab import files
    if os.path.exists("clothseg_code.zip"):
        os.remove("clothseg_code.zip")   # otherwise Colab renames the new upload
    print(f"Drive has code version {code_version(f'{DRIVE_ROOT}/code')}, this notebook needs version {CODE_VERSION}.")
    print("Choose the NEW clothseg_code.zip from the project folder on your computer.")
    uploaded = files.upload()
    assert "clothseg_code.zip" in uploaded, f"Expected a file named clothseg_code.zip, got {list(uploaded)}."
    for folder in (CODE_DIR, f"{DRIVE_ROOT}/code"):   # the Drive copy is used by later sessions
        shutil.rmtree(folder, ignore_errors=True)
        with zipfile.ZipFile("clothseg_code.zip") as z:
            z.extractall(folder)
    print("code uploaded and saved to Drive")

assert code_version(CODE_DIR) == CODE_VERSION, (
    f"The code is version {code_version(CODE_DIR)} but this notebook needs version {CODE_VERSION}. "
    "You uploaded an old clothseg_code.zip. Run this cell again and choose the new one.")
for needed in ("scripts/train.py", "scripts/resume_test.py", "src/clothseg/engine.py", "requirements.txt"):
    assert os.path.exists(f"{CODE_DIR}/{needed}"), f"{needed} is missing from {CODE_DIR}. Check the upload or the repository."
os.chdir(CODE_DIR)
print("code ready in", os.getcwd(), "| version", code_version(CODE_DIR))

Drive has code version 13, this notebook needs version 14.
Choose the NEW clothseg_code.zip from the project folder on your computer.


Saving clothseg_code.zip to clothseg_code.zip
code uploaded and saved to Drive
code ready in /content/clothes-segmentation | version 14


## 4. Install requirements and prepare the dataset

In [6]:
!pip install -q -r requirements.txt
import segmentation_models_pytorch, timm, cv2
print("segmentation_models_pytorch", segmentation_models_pytorch.__version__, "| timm", timm.__version__, "| opencv", cv2.__version__)
!python scripts/prepare_data.py --drive-root "{DRIVE_ROOT}" --height {HEIGHT} --width {WIDTH}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 14.4 MB/s eta 0:00:00
segmentation_models_pytorch 0.5.0 | timm 1.0.30 | opencv 4.14.0
[prepare] cache name: atr_576x384 | local: /content/data/atr_576x384 | Drive archive: /content/drive/MyDrive/clothseg/data/cache/atr_576x384.tar
[prepare] free local disk: 193.3 GB
[prepare] found archive on Drive (1.6 GB). Copying and extracting...
[storage] copied atr_576x384.tar (1.6 GB) -> /content/data in 23s
[storage] extracted atr_576x384.tar -> /content/data in 14s
[prepare] done in 39s
[splits] moved 56 val/test near-duplicates of training images into train. New sizes: train=16038, val=690, test=960


## 5. Merged architecture: SegFormer-B3 encoder + ConvNeXt-Tiny encoder with a fusion decoder

One network with two encoders. Both read the photo; their features are joined at each of the four scales; a new decoder predicts the labels. Trained for 12 epochs with the same data, seed and settings as the single models from the comparison, then printed next to them. About 3 hours on the L4. Only cells 1 to 4 are needed before it. If the session ends, run cells 1 to 4 and this cell again.

In [5]:
import os
import pandas as pd

DUAL_RUN = "abl_dual_b3_convnext_tiny"
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
assert gpu_gb >= 20, (f"This GPU has {gpu_gb:.0f} GB. Training needs the L4 (about 22 GB). "
                      "Choose Runtime > Change runtime type > L4 GPU, then run cells 1 to 4 and this cell again.")

os.chdir(CODE_DIR)
budget = int(minutes_left())
!python scripts/train.py --drive-root "{DRIVE_ROOT}" --run-name {DUAL_RUN} --model dual_b3_convnext_tiny --height {HEIGHT} --width {WIDTH} --epochs 12 --time-budget-min {budget}

COLS = ["garment_miou", "miou", "iou_upper-clothes", "iou_pants", "iou_skirt", "iou_dress", "iou_belt", "iou_scarf", "iou_sunglasses", "val_loss", "train_min"]
CORE = ["iou_upper-clothes", "iou_pants", "iou_skirt", "iou_dress"]
RUNS = {"SegFormer-B3": "r1_segformer_b3", "UPerNet+ConvNeXt-T": "r1_upernet_convnext_tiny", "merged": DUAL_RUN}
logs = {k: pd.read_csv(f"{DRIVE_ROOT}/runs/{v}/val_log.csv").set_index("epoch") for k, v in RUNS.items()
        if os.path.exists(f"{DRIVE_ROOT}/runs/{v}/val_log.csv")}
assert "merged" in logs, "The merged model has not finished an epoch yet. Read the training output above for the reason."
print()
print("garment mIoU per epoch")
print(pd.DataFrame({k: v["garment_miou"] for k, v in logs.items()}).round(4).to_string())
done = len(logs["merged"])
if done < 12:
    print()
    print(f"NOT FINISHED: {done} of 12 epochs. Run this cell again to continue.")
else:
    print()
    print("validation at epoch 12")
    print(pd.DataFrame({k: v.loc[12, COLS] for k, v in logs.items()}).round(4).to_string())
    base, new = logs["SegFormer-B3"].loc[12], logs["merged"].loc[12]
    gain = new["garment_miou"] - base["garment_miou"]
    worst = min(new[c] - base[c] for c in CORE)
    ok = gain >= 0.005 and worst >= -0.01
    print()
    print(f"merged against SegFormer-B3: garment mIoU {100 * gain:+.2f} points; largest drop on a core garment {100 * worst:+.2f} points -> "
          + ("MEETS the rule" if ok else "does NOT meet the rule") + " (at least +0.50, and no core garment below -1.00)")

config.json: 100% 70.0k/70.0k [00:00<00:00, 81.2MB/s]

pytorch_model.bin: downloading bytes:  69% 123M/179M [00:02<00:00, 125MB/s, 7.90MB/s  ]  
pytorch_model.bin: reconstructing file:  54% 96.0M/179M [00:02<00:01, 42.2MB/s]
pytorch_model.bin: reconstructing file:  58% 104M/179M [00:02<00:01, 44.4MB/s, 9.05MB/s  ]
pytorch_model.bin: reconstructing file:  62% 111M/179M [00:02<00:01, 46.1MB/s, 9.73MB/s  ]
pytorch_model.bin: downloading bytes:  89% 159M/179M [00:02<00:00, 110MB/s, 14.0MB/s  ]
pytorch_model.bin: downloading bytes: 100% 168M/168M [00:02<00:00, 59.9MB/s, 15.0MB/s  ]
pytorch_model.bin: reconstructing file: 100% 179M/179M [00:02<00:00, 63.6MB/s, 16.2MB/s  ]
Loading weights: 100% 628/628 [00:00<00:00, 21604.16it/s]

model.safetensors: downloading bytes:   0% 0.00/114M [00:00<?, ?B/s]
model.safetensors: downloading bytes:  81% 92.1M/114M [00:01<00:00, 91.6MB/s, 6.03MB/s  ]
model.safetensors: downloading bytes: 100% 109M/109M [00:01<00:00, 68.0MB/s, 10.3MB/s  ]
model.safetensors:

In [7]:
import json, os, subprocess, sys
import pandas as pd

STOP_AT = 20     # first 20; set to 30 and run again only if the epoch-20 check below says CONTINUE
SHORT_RUN = "abl_dual_b3_convnext_tiny"        # finished 12-epoch run, only read
LONG_RUN = "ext_dual_b3_convnext_tiny_e30"      # epochs 13 to 30, on the 30-epoch schedule
BASE_RUN = "final_segformer_b3_e30"
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
assert gpu_gb >= 20, (f"This GPU has {gpu_gb:.0f} GB. Training needs the L4 (about 22 GB). "
                      "Choose Runtime > Change runtime type > L4 GPU, then run cells 1 to 4 and this cell again.")

os.chdir(CODE_DIR)
budget = int(minutes_left())
!python scripts/train.py --drive-root "{DRIVE_ROOT}" --run-name {LONG_RUN} --extend-from {SHORT_RUN} --model dual_b3_convnext_tiny --height {HEIGHT} --width {WIDTH} --epochs 30 --stop-after-epoch {STOP_AT} --time-budget-min {budget}

assert os.path.exists(f"{DRIVE_ROOT}/runs/{LONG_RUN}/val_log.csv"), "The continuation has not finished an epoch yet. Read the training output above for the reason."
base = pd.read_csv(f"{DRIVE_ROOT}/runs/{BASE_RUN}/val_log.csv").set_index("epoch")
merged = pd.concat([pd.read_csv(f"{DRIVE_ROOT}/runs/{SHORT_RUN}/val_log.csv").set_index("epoch"),
                    pd.read_csv(f"{DRIVE_ROOT}/runs/{LONG_RUN}/val_log.csv").set_index("epoch")])
done = int(merged.index.max())
print()
print("garment mIoU per epoch, 576x384. Merged: epochs 1-12 from the 12-epoch run, 13 onwards on the 30-epoch schedule")
print(pd.DataFrame({"SegFormer-B3 (30 ep)": base["garment_miou"], "merged": merged["garment_miou"]}).round(4).to_string())
CORE = ["upper-clothes", "pants", "skirt", "dress"]

if done < 20:
    print()
    print(f"NOT FINISHED: {done} of 20 epochs. Run cells 1 to 4 and this cell again to continue.")
elif done < 30:
    # Check at epoch 20, fixed before the result (PLAN.md section 38): both models are on the same
    # 30-epoch schedule here, so their epochs 18-20 are compared directly. Three epochs are averaged
    # because single epochs move by about 0.5 points.
    m, b = merged.loc[18:20], base.loc[18:20]
    cols = ["garment_miou", "miou"] + [f"iou_{c}" for c in CORE] + ["iou_belt", "iou_scarf", "iou_sunglasses"]
    table = pd.DataFrame({"SegFormer-B3": b[cols].mean(), "merged": m[cols].mean()})
    table["difference"] = table["merged"] - table["SegFormer-B3"]
    print()
    print("EPOCH-20 CHECK: mean of epochs 18-20, validation 576x384")
    print(table.round(4).to_string())
    gain = table.loc["garment_miou", "difference"]
    worst = min(table.loc[f"iou_{c}", "difference"] for c in CORE)
    go = gain >= 0.005 and worst >= -0.01
    print()
    print(f"garment mIoU {100 * gain:+.2f} points; largest drop on a core garment {100 * worst:+.2f} points -> "
          + ("CONTINUE: set STOP_AT = 30 and run cells 1 to 4 and this cell again" if go else "STOP here: the merged model is not adopted")
          + " (rule: at least +0.50, and no core garment below -1.00)")
else:
    WATCH = CORE + ["belt", "scarf", "sunglasses", "shoes"]
    rows = {}
    for name, run in (("SegFormer-B3", BASE_RUN), ("merged", LONG_RUN)):
        out = f"{DRIVE_ROOT}/reports/eval_{run}_val_flip"
        print("evaluating", name, "on validation, original size, flip ...", flush=True)
        r = subprocess.run([sys.executable, "scripts/evaluate.py", "--checkpoint", f"{DRIVE_ROOT}/runs/{run}/checkpoints/best.pt",
                            "--split", "val", "--out-dir", out, "--flip-tta"], capture_output=True, text=True)
        assert r.returncode == 0, "The evaluation failed. Send me this message: " + (r.stdout + r.stderr)[-3000:]
        mt = json.load(open(f"{out}/metrics.json"))
        pc = pd.read_csv(f"{out}/per_class.csv").set_index("class")["iou"]
        rows[name] = {"garment_miou": mt["garment_miou"], "miou": mt["miou"], "core_garment_miou": mt["core_garment_miou"],
                      "boundary_miou": mt["boundary_miou"], **{c: pc[c] for c in WATCH}, "ms_per_image": mt["ms_per_image"]}
    df = pd.DataFrame(rows)
    df["difference"] = df["merged"] - df["SegFormer-B3"]
    print()
    print("VALIDATION, original resolution, flip averaging, best.pt of each run")
    print(df.round(4).to_string())
    gain = df.loc["garment_miou", "difference"]
    worst = min(df.loc[c, "difference"] for c in CORE)
    ok = df.loc["garment_miou", "merged"] >= 0.7491 and worst >= -0.01
    print()
    print(f"merged against SegFormer-B3: garment mIoU {100 * gain:+.2f} points; largest drop on a core garment {100 * worst:+.2f} points -> "
          + ("MEETS the rule" if ok else "does NOT meet the rule") + " (garment mIoU at least 0.7491, and no core garment below -1.00)")


[checkpoint] loaded step_00012024.pt
config.json: 100% 70.0k/70.0k [00:00<00:00, 33.6MB/s]
[model] dual_b3_convnext_tiny: 83.2 M parameters (encoder 71.9 M, head 11.3 M) | 15 classes | weights not initialised from ImageNet
[train] EXTENDED: continues run abl_dual_b3_convnext_tiny after its epoch 12, now with a 30-epoch schedule. Learning rate restarts at x0.600 of the base rate and falls to zero at epoch 30. best.pt starts as the best model of abl_dual_b3_convnext_tiny.
[train] RESUMED from epoch 13, batch 0/1002 (global step 12024/30060); best mIoU so far 0.7812; 198 min trained in earlier sessions; model weights restored from the checkpoint
[train] dual_b3_convnext_tiny | 15 classes | 576x384 | batch 16 | fp16 True | NVIDIA L4 | 8 workers
[train] data: train 16038, val 690 images (fingerprint 8b4d7a60ccb0) | 1002 steps/epoch
[train] schedule: 30 epochs = 30060 steps; this command runs until the end of epoch 20; time budget 592 min; checkpoint every 10 min
[train] epoch 13/30 step 26/

In [8]:
import json, os, subprocess, sys
import pandas as pd

os.chdir(CODE_DIR)
R = f"{DRIVE_ROOT}/runs"
B3 = f"{R}/final_segformer_b3_e30/checkpoints/best.pt"
UPER = f"{R}/r1_upernet_convnext_tiny/checkpoints/best.pt"
CORE = ["upper-clothes", "pants", "skirt", "dress"]
WATCH = CORE + ["belt", "scarf", "sunglasses", "shoes"]

rows = {}
for name, ckpts in (("B3 alone", [B3]), ("UPerNet alone", [UPER]), ("B3 + UPerNet averaged", [B3, UPER])):
    out = f"{DRIVE_ROOT}/reports/ens_test_" + str(len(rows))
    print("evaluating", name, "...", flush=True)
    cmd = [sys.executable, "scripts/evaluate.py", "--checkpoint", ckpts[0], "--split", "val", "--out-dir", out, "--flip-tta"]
    if len(ckpts) > 1:
        cmd += ["--ensemble-with"] + ckpts[1:]
    r = subprocess.run(cmd, capture_output=True, text=True)
    assert r.returncode == 0, "The evaluation failed. Send me this message: " + (r.stdout + r.stderr)[-3000:]
    m = json.load(open(f"{out}/metrics.json"))
    pc = pd.read_csv(f"{out}/per_class.csv").set_index("class")["iou"]
    rows[name] = {"garment_miou": m["garment_miou"], "miou": m["miou"], "boundary_miou": m["boundary_miou"],
                  **{c: pc[c] for c in WATCH}}

df = pd.DataFrame(rows)
df["average minus B3"] = df["B3 + UPerNet averaged"] - df["B3 alone"]
print()
print("VALIDATION, original resolution, flip averaging")
print(df.round(4).to_string())
gain = df.loc["garment_miou", "average minus B3"]
print()
print(f"garment mIoU of the average over B3 alone: {100 * gain:+.2f} points -> "
      + ("the encoders carry DIFFERENT garment information (fusion is wasting it)" if gain >= 0.005
         else "the encoders carry little different garment information (no fusion design can add much)")
      + " (threshold +0.50, PLAN.md section 39.4)")


evaluating B3 alone ...
evaluating UPerNet alone ...
evaluating B3 + UPerNet averaged ...

VALIDATION, original resolution, flip averaging
               B3 alone  UPerNet alone  B3 + UPerNet averaged  average minus B3
garment_miou     0.7441         0.7198                 0.7426           -0.0015
miou             0.7888         0.7717                 0.7885           -0.0004
boundary_miou    0.7349         0.7201                 0.7364            0.0014
upper-clothes    0.8523         0.8392                 0.8553            0.0029
pants            0.8379         0.8290                 0.8441            0.0062
skirt            0.7475         0.7110                 0.7377           -0.0099
dress            0.7237         0.7026                 0.7233           -0.0004
belt             0.5856         0.5658                 0.5949            0.0093
scarf            0.6600         0.5392                 0.6188           -0.0412
sunglasses       0.6621         0.6597                 0.6689